# Section 5: Calibration

Calibration of the three-country model to the 2025 US–China tariff episode.
Model-implied bilateral exchange rate changes are compared against realized data.

**Notebook covers:**
- Section 5.1–5.3 (main text): three third-country configurations (EU σ=8, Vietnam σ=8, ROW σ=2)
- Appendix B (extended calibrations): eight configurations used in the dashboard and appendix tables

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from tariff_exchange_rates import CALIBRATIONS, TARIFF_REGIMES, solve_3country, free_trade
from tariff_exchange_rates.parameters import make_params_3country
from tariff_exchange_rates.tariffs import make_tariff_matrix

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

# Realized FX data (Yahoo Finance monthly averages via fetch_fx_data.py)
with open('../data/fx_data.json') as f:
    _fx = json.load(f)['rates']

def d_AB(regime):
    """Log-pt change x100 in USD/RMB. Positive = USD depreciates vs RMB."""
    return _fx['RMB']['log_changes'][regime]

def d_AC(country_key, regime):
    """Log-pt change x100 in USD/C. Positive = USD depreciates vs C."""
    return _fx[country_key]['log_changes'][regime]

def d_BC(country_key, regime):
    """Cross rate RMB/C = d_AC - d_AB (log arithmetic)."""
    return d_AC(country_key, regime) - d_AB(regime)

## 5.1 Calibration Strategy

**Expenditure shares** from 2024 world goods export shares (WTO), fixing $\alpha_N = 0.60$:
$$\alpha_{T_j} = 0.40 \times \frac{s_j}{s_A + s_B + s_C}$$

**Labor endowments** proportional to 2024 PPP GDP (IMF WEO), normalising $L_A = 1$ (US).
Gives $L_B = 1.21$ (China), $L_C^{\text{EU}} = 0.86$, $L_C^{\text{VNM}} = 0.055$, $L_C^{\text{ROW}} = 3.51$.

**Elasticity of substitution** $\sigma$ reflects substitutability between third-country and Chinese exports:

| Config | $\sigma$ | Motivation |
|---|---|---|
| US–China–EU | 8 | *China Shock 2.0*: Chinese manufacturing competes with European high-tech; Broda & Weinstein (2006) |
| US–China–VNM | 8 | Vietnam as China re-export hub; Vietnamese exports are largely Chinese supply-chain goods |
| US–China–ROW | 2 | Aggregate rest-of-world; low substitutability by construction |

In [ ]:
# Table 2 (main text): calibration parameters for the three main-text configurations
print(f"{'Configuration':22s}  {'aTA':>6}  {'aTB':>6}  {'aTC':>6}  {'aN':>6}  {'LC/LA':>7}  {'sigma':>6}")
for name, p in CALIBRATIONS.items():
    aT    = p['alpha_T']
    lc_la = p['labor'][2] / p['labor'][0]
    print(f"{name:22s}  {aT[0]:6.3f}  {aT[1]:6.3f}  {aT[2]:6.3f}  {p['alpha_N']:6.3f}  {lc_la:7.3f}  {p['sigma']:6.1f}")

## 5.2 Tariff Regimes

| Regime | Description | $\tau_{AB}$ (US→China) | $\tau_{BA}$ (China→US) | $\tau_{AC}$ (US→C) |
|---|---|---|---|---|
| 1 (Fentanyl) | Unilateral US tariff on China, no retaliation | 20% | 0% | 0% |
| 2 (Peak escalation) | Bilateral escalation + universal baseline on C | 145% | 125% | 10% |

Realized data: log-point changes ×100 relative to 2024 annual average (Yahoo Finance).  
Regime 1 → March 2025; Regime 2 → April 2025.  
ROW config uses the Fed H.10 Advanced Economy trade-weighted dollar index.

## 5.3 Results — Main-Text Configurations

In [ ]:
# EU and VNM data from fx_data.json; ROW uses Fed H.10 advanced-economy index (hardcoded)
ROW_REALIZED = {
    'regime1': (-1.05, -0.31),
    'regime2': ( 2.18,  3.57),
}

fx_key = {'US\u2013China\u2013EU': 'EU', 'US\u2013China\u2013VNM': 'VNM'}
rkey   = {'Regime 1 (Fentanyl)': 'regime1', 'Regime 2 (Peak trade war)': 'regime2'}

rows = []
for cfg_name, params in CALIBRATIONS.items():
    eq_ft = solve_3country(params, free_trade())
    for regime_name, T in TARIFF_REGIMES.items():
        rk   = rkey[regime_name]
        eq   = solve_3country(params, T)
        m_ab = 100 * (eq['e_AB'] / eq_ft['e_AB'] - 1)
        m_ac = 100 * (eq['e_AC'] / eq_ft['e_AC'] - 1)
        m_bc = 100 * (eq['e_BC'] / eq_ft['e_BC'] - 1)
        dab  = d_AB(rk)
        if cfg_name in fx_key:
            ck  = fx_key[cfg_name]
            dac = d_AC(ck, rk)
            dbc = d_BC(ck, rk)
        else:
            dac, dbc = ROW_REALIZED[rk]
        rows.append(dict(Config=cfg_name, Regime=rk,
                         m_AB=m_ab, m_AC=m_ac, m_BC=m_bc,
                         d_AB=dab,  d_AC=dac,  d_BC=dbc))

df = pd.DataFrame(rows)

hdr = f"{'Config':22s}  {'Rg':2s}  {'Mdl dAB':>8}  {'Dat dAB':>8}  {'Mdl dAC':>8}  {'Dat dAC':>8}  {'Mdl dBC':>8}  {'Dat dBC':>8}"
print(hdr)
print('-' * len(hdr))
for _, r in df.iterrows():
    rg = 'R1' if r['Regime'] == 'regime1' else 'R2'
    print(f"{r['Config']:22s}  {rg:2s}  {r['m_AB']:>+8.2f}  {r['d_AB']:>+8.2f}  {r['m_AC']:>+8.2f}  {r['d_AC']:>+8.2f}  {r['m_BC']:>+8.2f}  {r['d_BC']:>+8.2f}")

In [ ]:
# Figure 1 (main text): model vs data bar chart for Regime 2
cfgs    = ['US\u2013China\u2013EU', 'US\u2013China\u2013VNM', 'US\u2013China\u2013ROW']
clabels = ['EU', 'Vietnam', 'ROW']
er_keys = [('m_AB', 'd_AB', r'$\Delta e_{AB}$'),
           ('m_AC', 'd_AC', r'$\Delta e_{AC}$'),
           ('m_BC', 'd_BC', r'$\Delta e_{BC}$')]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, cfg, clabel in zip(axes, cfgs, clabels):
    row        = df[(df['Config'] == cfg) & (df['Regime'] == 'regime2')].iloc[0]
    model_vals = [row[mk] for mk, _, _ in er_keys]
    data_vals  = [row[dk] for _, dk, _ in er_keys]
    labels     = [lb      for _, _,  lb in er_keys]
    x, w = np.arange(3), 0.35
    ax.bar(x - w/2, model_vals, w, label='Model', color='#2166ac', alpha=0.85)
    ax.bar(x + w/2, data_vals,  w, label='Data',  color='#d6604d', alpha=0.85)
    ax.axhline(0, color='black', lw=0.7, ls='--')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(f'US\u2013China\u2013{clabel}', fontsize=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    if ax is axes[0]:
        ax.set_ylabel('% change (+ = first currency depreciates)', fontsize=9)

handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, lbls, loc='lower center', ncol=2, fontsize=10,
           frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Regime 2 (Peak Escalation): Model vs. Data', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('../output/calibration_results.pdf', bbox_inches='tight')
plt.show()

## Summary (Section 5.3)

1. **Regime 2 sign predictions are correct across all configurations.** The model predicts USD depreciation against third-country currencies in all three cases, consistent with the bilateral trade war mechanism of Section 4.

2. **The RMB depreciates against both the dollar and the third currency** in Regime 2: US tariffs on China (145%) substantially exceed Chinese retaliation (125%).

3. **Regime 1 magnitudes are overstated.** A 20% unilateral tariff moves exchange rates little in the data; the model abstracts from nominal rigidities, financial flows, and partial pass-through.

---
## Appendix B: Extended Calibrations

Eight third-country configurations reproducing Tables A2 and A3 in the paper appendix
and the numbers in `data/calibration_panel.json` (dashboard source).

| Country C | $\sigma$ | Rationale |
|---|---|---|
| EU | 4 | Moderate manufacturing competition; product differentiation limits substitutability |
| Japan | 3 | Advanced manufactures (autos, machinery); limited overlap with Chinese exports |
| South Korea | 5 | Electronics/semiconductors create meaningful overlap |
| Mexico | 5 | Nearshoring beneficiary; autos and electronics overlap |
| Canada | 2 | Commodity-heavy basket; low substitutability (Feenstra et al. 2018) |
| Vietnam | 8 | Assembly/re-export hub; close substitutes at product level (Broda & Weinstein 2006) |
| Taiwan | 3 | Semiconductor dominance; limited aggregate substitutability |
| India | 3 | Growing manufacturing base; limited current overlap |

In [ ]:
# Parameters mirror scripts/precompute_calibration_panel.py exactly
S_A, S_B = 0.084, 0.143
L_B_app  = 1.21

def make_alpha(s_C):
    denom = S_A + S_B + s_C
    a_A   = 0.40 * S_A / denom
    a_B   = 0.40 * S_B / denom
    a_C   = 0.40 - a_A - a_B
    return dict(alpha_T_A=a_A, alpha_T_B=a_B, alpha_T_C=a_C, alpha_N=0.60)

APP_CONFIGS = {
    'EU':  dict(s_C=0.120, L_C=25.1/29.2, sigma=4, fx_key='EU',  label='European Union'),
    'JPN': dict(s_C=0.033, L_C=6.6/29.2,  sigma=3, fx_key='JPN', label='Japan'),
    'KOR': dict(s_C=0.027, L_C=3.0/29.2,  sigma=5, fx_key='KOR', label='South Korea'),
    'MEX': dict(s_C=0.028, L_C=3.3/29.2,  sigma=5, fx_key='MEX', label='Mexico'),
    'CAN': dict(s_C=0.024, L_C=2.5/29.2,  sigma=2, fx_key='CAN', label='Canada'),
    'VNM': dict(s_C=0.016, L_C=1.6/29.2,  sigma=8, fx_key='VNM', label='Vietnam'),
    'TWN': dict(s_C=0.022, L_C=1.7/29.2,  sigma=3, fx_key='TWN', label='Taiwan'),
    'IND': dict(s_C=0.019, L_C=14.5/29.2, sigma=3, fx_key='IND', label='India'),
}

tau_ft_app = free_trade()
tau_r1_app = make_tariff_matrix(tau_AB=0.20)
tau_r2_app = make_tariff_matrix(tau_AB=1.45, tau_BA=1.25, tau_AC=0.10)

def solve_app(cfg, tau):
    alpha  = make_alpha(cfg['s_C'])
    params = make_params_3country(**alpha, labor=(1.0, L_B_app, cfg['L_C']), sigma=cfg['sigma'])
    eq_ft  = solve_3country(params, tau_ft_app)
    eq     = solve_3country(params, tau, init=[eq_ft['log_e_AB'], eq_ft['log_e_AC']])
    return {
        'm_AB': 100 * (eq['e_AB'] / eq_ft['e_AB'] - 1),
        'm_AC': 100 * (eq['e_AC'] / eq_ft['e_AC'] - 1),
        'm_BC': 100 * (eq['e_BC'] / eq_ft['e_BC'] - 1),
    }

print(f"{'Country':14s}  {'sigma':>5}  {'aTA':>7}  {'aTB':>7}  {'aTC':>7}  {'LC/LA':>7}")
for k, cfg in APP_CONFIGS.items():
    a = make_alpha(cfg['s_C'])
    print(f"{cfg['label']:14s}  {cfg['sigma']:>5}  {a['alpha_T_A']:7.4f}  {a['alpha_T_B']:7.4f}  {a['alpha_T_C']:7.4f}  {cfg['L_C']:7.4f}")

In [ ]:
# Table A2: Regime 1 (Fentanyl tariffs, March 2025)
print("Regime 1  (tau_AB=0.20, isolated)")
print(f"{'Country':14s}  {'Mdl dAB':>8}  {'Dat dAB':>8}  {'Mdl dAC':>8}  {'Dat dAC':>8}  {'Mdl dBC':>8}  {'Dat dBC':>8}")
print('-' * 78)
for k, cfg in APP_CONFIGS.items():
    m  = solve_app(cfg, tau_r1_app)
    rk = 'regime1'
    print(f"{cfg['label']:14s}  {m['m_AB']:>+8.2f}  {d_AB(rk):>+8.2f}  {m['m_AC']:>+8.2f}  {d_AC(cfg['fx_key'], rk):>+8.2f}  {m['m_BC']:>+8.2f}  {d_BC(cfg['fx_key'], rk):>+8.2f}")

In [ ]:
# Table A3: Regime 2 (Peak escalation, April 2025)
print("Regime 2  (tau_AB=1.45, tau_BA=1.25, tau_AC=0.10)")
print(f"{'Country':14s}  {'Mdl dAB':>8}  {'Dat dAB':>8}  {'Mdl dAC':>8}  {'Dat dAC':>8}  {'Mdl dBC':>8}  {'Dat dBC':>8}")
print('-' * 78)
for k, cfg in APP_CONFIGS.items():
    m  = solve_app(cfg, tau_r2_app)
    rk = 'regime2'
    print(f"{cfg['label']:14s}  {m['m_AB']:>+8.2f}  {d_AB(rk):>+8.2f}  {m['m_AC']:>+8.2f}  {d_AC(cfg['fx_key'], rk):>+8.2f}  {m['m_BC']:>+8.2f}  {d_BC(cfg['fx_key'], rk):>+8.2f}")